# Train Ball Detector

This notebook trains a one-class YOLO detector from datasets exported by `tools/export_yolo/export_ball_dataset.py`.

Expected input after export:

```txt
data/exports/ball_yolo/
  dataset.yaml
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```

## 1. Runtime Check

In Colab, choose `Runtime > Change runtime type > GPU` before training.

In [ ]:
import sys
import platform

print("python", sys.version)
print("platform", platform.platform())

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet")

## 2. Install Training Dependencies

In [ ]:
%pip install -q ultralytics

## 3. Provide Dataset

Upload `data/exports/ball_yolo.zip`. This cell finds the ZIP, extracts it with progress, repairs the dataset path for Colab, and verifies the train/validation files.

In [ ]:
from pathlib import Path
import shutil
import zipfile
from IPython.display import clear_output

DATASET_ZIP = "ball_yolo.zip"
extract_dir = Path("/content/ball_yolo")

def find_existing_zip():
    candidates = [Path(DATASET_ZIP), Path("/content") / DATASET_ZIP]
    candidates.extend(Path("/content").glob("ball_yolo*.zip"))
    return next((path for path in candidates if path.exists()), None)

zip_path = find_existing_zip()
if zip_path is None:
    from google.colab import files
    print(f"Choose {DATASET_ZIP}.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if DATASET_ZIP in uploaded:
        zip_path = Path(DATASET_ZIP)
    elif len(zip_names) == 1:
        zip_path = Path(zip_names[0])
    else:
        raise FileNotFoundError(f"Expected {DATASET_ZIP} or one ZIP file, got: {list(uploaded)}")

if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as archive:
    members = archive.infolist()
    for index, member in enumerate(members, start=1):
        archive.extract(member, extract_dir)
        if index == len(members) or index % 50 == 0:
            clear_output(wait=True)
            print(f"extracting files: {index}/{len(members)}")

matches = sorted(extract_dir.rglob("dataset.yaml"))
if not matches:
    raise FileNotFoundError(f"No dataset.yaml found after extracting {zip_path}")

DATASET_YAML = str(matches[0])
dataset_root = Path(DATASET_YAML).parent.resolve()
Path(DATASET_YAML).write_text(
    f"path: {dataset_root.as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: ball\n",
    encoding="utf-8",
)

for split in ("train", "val"):
    image_count = len(list((dataset_root / "images" / split).glob("*")))
    label_count = len(list((dataset_root / "labels" / split).glob("*.txt")))
    print(split, "images", image_count, "labels", label_count)
    if image_count == 0 or image_count != label_count:
        raise RuntimeError(f"Invalid {split} dataset: {image_count} images, {label_count} labels")

print("dataset yaml:", DATASET_YAML)
print(Path(DATASET_YAML).read_text())

## 4. Train Baseline

Start with the nano model. For a tiny fast ball, `imgsz=960` or `1280` is usually more useful than `640`, but it uses more memory.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=960,
    batch=4,
    workers=2,
    cache=False,
    patience=25,
    project="/content/runs/ar_ping_pong",
    name="ball_yolo26n_img960",
)

## 5. Validate And Inspect Predictions

In [ ]:
from pathlib import Path
import shutil

best = Path("/content/runs/ar_ping_pong/ball_yolo26n_img960/weights/best.pt")
model = YOLO(str(best))
metrics = model.val(data=DATASET_YAML, imgsz=960)
print(metrics)

download_model = Path("/content/ball_yolo26n_img960.pt")
shutil.copy2(best, download_model)
print("model ready to download:", download_model)

from google.colab import files
files.download(str(download_model))

## 6. Optional Larger Comparison

Run this after the baseline works. It may need a better Colab GPU.

In [ ]:
# from ultralytics import YOLO
# model = YOLO("yolo26s.pt")
# results = model.train(
#     data=DATASET_YAML,
#     epochs=100,
#     imgsz=960,
#     batch=-1,
#     patience=25,
#     project="/content/runs/ar_ping_pong",
#     name="ball_yolo26s_img960",
# )